# V8 structured notice checker — development comparison
This revision uses the 500 cases for development; results are not held-out accuracy. Both versions run fresh with shared models. No pretrained weights are fine-tuned.


In [ ]:
# V8 development comparison: approved 500 cases are now reused development data.
# Frozen V7 and V8 run fresh with identical loaded model objects and thresholds.
!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 sentencepiece
import re, pandas as pd, numpy as np, torch
import base64, zlib, json, csv, io, hashlib, time, platform, importlib.metadata, types, sys
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML
from collections import Counter
torch.set_num_threads(2)
torch.manual_seed(8)
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)

NLI_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(device).eval()
# find which output position means "contradiction" from the model's own label list
CONTRA = [i for i, name in nli_model.config.id2label.items() if "contra" in name.lower()][0]
print("NLI labels:", nli_model.config.id2label)

def similarity(zh, en):
    a, b = labse.encode([zh, en], convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(a, b))

def contradiction(zh, en):
    # probability that the two sentences contradict each other (higher of the two directions)
    scores = []
    for premise, hypothesis in [(zh, en), (en, zh)]:
        inputs = nli_tok(premise, hypothesis, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(nli_model(**inputs).logits[0], -1)
        scores.append(float(probs[CONTRA]))
    return max(scores)
frozen=json.loads(zlib.decompress(base64.b64decode('...')))
raw=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(raw).hexdigest()=='2bad8349bcfaf4b48358b7aa23e1fa19321e45a8c38332ea9cdd01bfeccc165d'
references=list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
assert len(references)==len({r['case_id'] for r in references})==500
# Only bilingual input and identifier pass to the prediction loop.
inputs=[{k:r[k] for k in ('case_id','chinese','english')} for r in references]
patch=zlib.decompress(base64.b64decode('eNq1O2tzG1WW3/UrLp7aavW4oyJs7S6lRM6kQjJkGEwqMVQtjtfVUl9ZPWl1a/qh2FaryjyS2BiT7GISEgyEHQIJjxAIDxND+C+sW5Y/zV/Yc869/ZJkwzDAw+q+j3PP+3Hv7bGxseceZ1UnsA1uaOx8w7H4AdvxzRpnHm/qNjyxOVdvNnW3VCic0l2Pe4zrtQazdHsu0Oc4M2Fui8Mf27cWSmzSYVVu1xow4xw7+YSnwcgqt+CXz+vNlgXzHbfg6r7p2Dq+6S5nNcf2AsvnRokdb3N3gbV03+euzZqB50MvtDG/wRmsYcJwgeEhFtjnbOe8XahZeuARYILku0ENwXusrlsWq+q1c8x3CEDddRa5zWoNXjvH3RKbapgeg/90Vjdt0+cFSbykWcOl2By3uatbzHd127MIc4Yccz1ftw3TnsOFz3NzruEjGH7AD2xoLRXGxsYKZrPluD5zefwEfTUH2K37egHQaTJ8Ago8pECOSZoKhT+kz/QXGIwYlgsM/jkHzC8zIJjefKdl1tLXul7zvTIzzJovuvm8L3oLv2NTOFYw3wbxs1bD1QEBQbDuVk0gFuTgAcdBmtwDwcxDE4tZTYLBsVUO0LzzwGjnPDeYjrwkRBjwBnnu8uoC423d4CQB04Ym4LSRCKFwfIpVmKsUj5w6TDMnpv+rpB0qP3LkrD0zfkRVCs8PD9D+b+nF7JCnn5mcenJ28ujTx8/AWOVPuh0g/id41aWHp3UXlPZoyzUteF5gfwpsDn+sBXY0mENazvCWz5tVULRnar6Dv5OgdtTwBK/Rg1LyWpbpF1W52tNHT8FaHa+EpLtFtczM8YOsDspgaiBKG9Q1aILm+LyYQU/tyumnjk5NHT89ieiGSukvjmnnhhUKBq8Di91mESWnCom73A9cO6tEJRyiW+YiLyqTJ546pmg0vOTylqXXoPHv372qaPBvrukGNJShCdTBbBXlYjXLqZ0rNjTW0K26xpqmHfi8MunYXK7eqJi2X2yoh+I+fBWPaAOPqjTKrJNmHGSHK6yBfw4+hr3Y9ii+ygmH2b8/WmaubnqcPadbAT/uuo5bVEy7DeQYAhtFzZJdVzqNfzn42HgRIMIqiGbMfOR3caw1po1tb65Ga8vw0Lt+Y3vzFWr5VLZ8+jd8UBk4JA7olh99zOiWOwIhelEkJ/xiy+V1c14jRyfJhyXxrVJRuK2Ipgxybp00tCMmdhsTZ41O56D2WLerFo+Uy9k+WJB6sU89ctb7fW4iUDUxrbdm1LOlI034XynsswoOBiqBrFAQHgqyw+jSq73rt0NBeygoV/dCcLp3/cXdrXdmANFReIpB0fJF9YhkkG82waatoqG1KoqSV06hRsZ0a1xpKDOaeAA88bk0x4G14wpAVlSpdqDFHBx+saXtyesh6rtGBrGz3rhsbU50csbVVUczDwam03sbyyOA9q7dUlL8JK1ZDJuVxA1ME41NZSbWxpk8/kLf0FjigUKtDX2hkrQacWtsP4crzcOV1HbgHcYfrvzrwf2sBpEdMpom6faBDsyPtfz0s38mV9kh/k7PaExZbOBDl2jWDaOI+GsYYjQZjjWIrhCKY19EIKZx1ExJb2EOUCzScJeXak6zZVq8KGeOu8o0eOzSzBEFe0+qCSgVo9EzLkRTjMPgoVxOQYdCiX/eYbbexBcR0SE+8TgZaDmWDpFkgTl1Bi4Kos45bpcK6H8RJ3QIhOG+toujBWzNwWBF9OLU6aICyGAwlqkDaLAFTrPmWBaHCbDAghO4kF0Ycxza6RdEBYQoqlZUvKDaNH0xxuV/Dbjn42yX6zRXt2HZGkRebJVP8eyZFD1C0YREQ/M4cMyoFGMEkoVTLAlDFSlFIoDSLE5CBYsCA55ZXCIa46jm1s6rAZjPCUcwv3N8qquB4+oI5nUZWhCtNKHXQeJhlQNruco6xNauouXg4j+W3qwaOjM0vSJIrFYEkeWOIqBK1MugwXqXCYisU+0qSKMxrYjuxOiAYlpb0gqzquksvat0BW3YmVcAIk7XqlLuWbZpyu7NN6PLV6MrazsrGEej5Xs7H9yNbr4Y3fwcnknWefnBlKurOx+vRpv3e8tfw2vv8pXtB+/vrN/vf7w6JF1Ub6/S0csDgt1XjENSrJYH9WD/+QNC7+4vdIgK0aUH0eanYXT5pf7ypbD/8IOdKxfVzvNTXejrbaxFq9fVIxBDUCWAyugCeVR9AtgeghRUbI8urmFjNWncTyfsCjFmSBVscLWKrszMjCuxaJVxaqxCYzfV3oSIvDiBGsBPyBTRF3KNHr7aW1kbFK0WYzOERV5WMSIDEhiJzICqSHQy+iJQESozwF+pQfuhlVOLGK9BYZPLPQ1JrdnyGUUN8KJl1nRsvwHOtO5xn5J3qgsheYbC0ISe3sYSILe9uQSREWohXEnm+6J4KBWQTHSv6JQEdD7fMt0FovMoixvrGR+CTl8ELkAYcnNR5Rno0/FR4ES6BDWl406AhZ5vmLVGSJYOzs5n53Us4byAGznWDKBQViRYGioAx35EwM45knid2GDk5LqDdQ46cAEAeElEQ+gcSTRKcOfGy731r3uv3+htrPTeWO5fur9z/TuwpwxVO298DUkI/Bc9eCMkJq8Cn6OlVWjC6dSh/rPkAVnZhX4F0gRdhCGS1tt4t3f1uz2oG6RLzepT9PnF3tYPvxqBP00aWMAkLOB5oN8HMOiYZEjnwZJgnCyDIUZDVe4KE6uDHdXrZs2E7IS1uNsERKCjVODzNagd4XG2jflYJUODZxr8gOGAvieDgBDP1+t1JhKdGkcLsGNUyDYcSKUAiZg6xC7VTapzlMC2YAKGmuvv7779Dsalyx9FG7cVNQ4QluOBeUKyZAvrzS6KDh8XalHAA1k+EbjInWxgx2dEnxH6gKOEiHJFjCYECqFYocFtFRyBWKQpymaYA3wzAsjQaBuAtGYqB3bQE/zza8jomnqiAe5DCj8gLzXPClLrjXd3r76uybqMMBF8DqO1+9vf/wCqu9J/4Zvo9feA5dELX+5eXQK9i5a+0zBSXvkgWrkUbm+tqaJr99rG7tWVrDsQ7T8Ddv/OF/3rb4yEPQCVwkSMdQhrqAOLCP1IoYOhZqFj/Xd1tbf+MMdBsv2fwUGwpj+6TtBiTtUy58QOGchpAYKD7+u1BshUZtFzOOwQ+Gzr3AHgeGpHKEoPykDcLslGEpoAocg5hyYRqxApDIQ5QMOLgXlnS0C7wKNteib0IPVNx9CtCdqvqnICxCnB1402+A0eGg4Zu80xmom2eLm8x4VkyGybRqBbyYpgzHLjK8F0AISCidlfA9OFIdKeCaHUoEvAKNf30EiLCqKZ2DCilUzujtLT125Fl6+B3kQ3v4SEpX/7I237+4vRxpu7H6+L95QBuzeXw95nkPksq7s3r+x8+TLxElxxtHxr+8GD6G8fR5c/y4LCBaBkk9BWLo0Etf0Acz0JcEhzBmTX+g25Ca4f8FH24BzmO47F0Z2bc3YTPXhTb3my8qNkx6n+BRM6UFNodFwN5s/proH+B32/2PcUiVapcPTY1DOnz8wen6woYlPWxz1x7nohbpwAHC/E8NWKDaFeD9uOBRkOhzFKPP35JyFeXf50e3Nre2srhJy0/79Xd9bfDVGsDy8IMw13Ht6Jvrkl5+BeQ6WTTsKUL7e+qFcEHOiMscEiJAOU2JhDDwbIhcpKBtNuxhDByZrcU2Q9AHVgeNRXcyGDfA1u6k50EgZ1VbHPSwMt0xOZJY40TNx7rAY+zw9P20UGCP7ANXXLy9kiQSLAVEyjxJQyxkpsSm1LU5LpKVySeDIjXS6dNsrWUgdL3nWY1uef7Kr9L272v/ikd3MzurcGlUV09a62B6mZ0V8+hIxpZ+VO//6lNLGipTBULF8EqUQfvAipU1mA3Fn/fPTqYl2wWQFMjttjcbGscAIbt3FN7WeB1/4B4MOhROpQ6ydlmaj7dCrVmZ8SZ25SVrBYGGZUGQIiRCWfBS3aLRu1r7HgBKB6C3KI2MaJp9kB5R5JaEXX7QRIA5R4QEPo2NYCowIlaROWgjUtJiwELineM7ygATSHY65GPgwiGQezzPm0kalhLpJIpJJgEk+Ni8JWA/fMJAdyuf4I7oh6nxQTanwIFL23vuxdvdd/4bOdr97bfWurf+dzLWGHCA04Yv3bEOIRvECMEO+qLHI/fB2U7VegPEuxWPfXITiLJpC7/cPbO29c395c6999RYPsC0MlZGdZrgxz5BeRJ4LVMcd1uTzDrOKGZC2ABls4Ty+ArAnSZCCN0i88avIgAFCpg/uuOiXz1Acple/lCnPHacKgGLxwOJhMCeLLoPpgOCL2QcyEwbSxZ+HO+LgKL2XwiVSk665looKbbi2wdBeUPteOgValjOxZkEYCSRJC0HIMiilEBJUymG5RbIzHHcqMquJmZUL70EBAEgft6byXQD7R3ZXea/8TRhv3drfeESJTy/2PVyHx3lm/PYhgdPd9zLp71+5EK2u7Szei/34l7C+/Aj877z9QeytL0fI18NFZDsEUzNNpGfVI9M0X+Hb5au9rPJ5Z233hDiyjZurnb0GPRizbX17pv/cquHJYTiy812qwRAI4rTE+3Opfuj8CLkL8eDVa/WH74ds7Fz5MiAH6oodLA6CH/feQ9rR+SxGO2MUHSBoMqsRHRghaEYcyWtKGUETbfjv/8mTCq0zDDFFx/2zDoOMHl7dNJ/Dicp1OXtAEO7B+lzQfKxUchTUt9UKN0wGMu2LXuNVygG7KRgcWTu0/VyKXRWCC0JGZKkpgAkuLnrK4Dha3B7K8qZsW8xpiO4RQVdSZEfviP4c9oGmiaOxduwUZC6giKPTOiw8IGS167d3o7g1aAhMZMgkYIqyBOLAP3OjeJahQCZBYAjRX2GF04ZNo8yWASJBz6v8TXJXVt6hWBMoxssIutreg7RPw3rsvfQU9vQdXNbFmYnz/8JrSdxDIISiCFbD29vc3sRn5KZBb/zamPpZOck5Bji2WTrkwetM+tUvNqmBzxjwpyJXTQ01pQ3nTHBokjEoV8emP8kYMhqNkMwDfA8OkPS08HsbTPNA2OynrISk6xxe4AfXTs09g8dRRTnEXCGtirQHVSvqiKcfnG2bVhLQOe3j6oikn7TbdH/FaXD8nZpqDTZpC+wCe2PuS1SOOlPU/VZdY3yAmUId1lP57t3vfrfeXLgwiEl1+Kfr8DdGRw2Nn7YvdF5f6n7wl+kbgACIW8hRF9Ijl08CcsM7D7ZLA4qkHilvOeuMx42l3lbRsQnddE2QU6obcSVHRTXRqfnFMXlwa08a4PaZ2Yf7Z38s67XzDmUiZH6bcDgfZG47ipEoYtLlbnRCnHiGhwUM8unVVpvuMMAAgNTPFIM5UcpE/vl4l6MFNgJiSZH9X9OQO+tIxIt2TbMCdg/i6FuigUk4vKsQdYLJKorSJyiplm8/7xbYwNK2NVibUtATsaHpFcbCaogB4ARNTpPJQJS14uu7T0UyKcHxkq8Rndih94h4Ok2R0sYRB/mbuFGQXGCJOcloZmf/Q7t7roOKgAtubS/3lTzIKFC3fi25+KYaE0YVb8KKWB/RnsTFCfzpj4Zi4tySsCJaOHVdO9DQ50RdxuBaKVeP1Ev+8B3bxOIElee1h/MCR7oMa1iSvruxef7l/e+uX4TiUDo2w2dYv02zQaLHKr6fOgvDpWE1nRqunXDWvn/L8M6OfghOj9fPn6yVEjjMm3gGlYbjzBJYFBuKV4wKI6VTBWNznyd1DcRgDboo2g2oNyrXMORMLmzQqyguj4gSfykKBi9yvwlRMPSLcKdWMiYuKfVMuI0mmJpvN2LPHpGz4Tyb+52CeNnjWMQxq8Owid/Umo1YDPI4tPjvcG8GIxBlI5Y+1HvKr6MNVYYHD7Tgjm+1BkzDCcm4wpO1D7NtnQTxoGFqw/9U3WGtTKUbLzozmLsIdyphyjiHBShNDk0A8iEpawsNikv+J90ySrNTkB5Kr0cKgC1QtvCs9K64T0wXO7NUxr5Le6xzI7vI3rZLyR1yzStO9ZkWOLNUDyyK7KHrphQN5haxZpk0r0w540mVUmiXKP/BicFE9JG7rYjijB9pUEXdCM8BkAQUsoJxPXiwZPNnMXefI7tPg7rw4qKTpijjHUghHTBBHoDkChDggkyC2t9b2nY83HGhHHw8KDYPOfDI3w8VVcXHq67GGCSPQedDRpA565oOT8izHL43gAl7oQxxcXvI43isuusrZKjiZmm6HTX0hxLOk8DyeR8JfJ/Qbuh/iyU4ozzNxfzCEwYB+KPfp5H0pca0hhPossAz40S0rxFFV3NIJWwBbXOYIobYzPRyKWmAbYct12oC7EQpxhKZdswI8ylbPVhWNJCsu3o1g1bB8s5Thac/uxlIIphnGJ5XyGHJzLXp4jX4uXQyjly7sXr8Cwz6iH5JVKDYbYMgH8P962FvfiDa/Dne+vb+99WYoLrmEGG2urIXb365GD1+ln5U1ifIobH13oSzutlekkYDVJb2CfFZMr0ZqT/EFetiD9PjC3+C9r4FLecQXcX1V0RXM/8RzVXZBI0IxRiwi72GKi/vihqQQCFGhpV4gGWhz4USEwgYuN2blxp7n2EUMgdzjGrfnLNNrSJ+yqPFKzunEw1CqqpbrkjM1VOXcndPF+LIp3edeLCGyj1R4SXxokMWPmI0qBxG6Mp2pUfkCmSTkDdwoesCixRLRqYb4wuWLmjqLxbbG2xU5ipgKMDTaFqXkoW5yAwjgPzEgK9PFNiDdzvujGNn4qiq5vxpkIHOOu1BBiJJh8hoHoiXYJBt4OxnhtXQbMCa3Hg+iNk5tak6cIBgtXryA34ScBEeEH6Y89zijL1t01sKkyk8+T6Grby0dUx+L623xhQlkPFhi1wNbujCTdq1L+JEJfUdCPY5jJV+RWG4wW8NzwcKsACzFP0vnKbPtx1lFnLvIdqFzs6i/QESgW7P4/Qx+PgNj99A6cF2VPaAPzSgIvzyV+woEtxgOuHg42+AB6Dd+caRbniO++qEbwZaJnwmBjZ0+89wpr8RwQ1mXwGwe+Jgd00dGzNXxFgikWenVHeAdp0+KDKg7LUCIci8bUzPLrNFVhVJMCabKCxBq0keMhelnE5k5KTjDrNcB0bKiMWVUI3hqPl85ATTxnFrAIoXCHxIZFZv6vGcu8sq/HXxMFVfs0UfMejUICx6yc7EBnJRs/x07AWEf9MZpOZ4MYmh9k38+KdLlOHkGldLnKMcusRPmPDD8uf8A/gDIhmMZXimLUdEzm6a4Uy0X08iZuTraCqxRVHB7bOnGzru3el+vRJdfUcYXG+MK1F3gL+lTKvn5FMyqURaKCa8yzu1xpZRc98+qXKwjszlTynzvQm7LqOzrCEdMjn2amI4iR6eVOgRS2/0VfS+oia4Y4HqwYJqFTKwBWqNIM4hhKYMBACUuPXXiEITLNkRWaDY1we/KkOxLImJw8Rub0lE8DGqRq01jBTMc7gknDn6miugzrK7Ai5FGyKuVeKkKsiff41Y9VgMduOrF3hwyDUygZgOUpY8beRVE8fCZk0/PTskECnCdAJ2bnZIITeIHT2DBSJyPjtmlSiTOqOgWjhe4bbOd+cAwrupKsdQwUEAKUoegA7zE7MoYVzTJA1Vlj1T2GBKzJxNd9nf7iiQyg7AyGAdi1ueDgWwdjAi5oXFIEDgl1A3xFX0byUviWs5ksZOQhBwQF2yZ0Af50Z7ParplVl26hmuZ53haOQvbAot/gu42ZaDFF4TBb54HN6u7TYjUFu7MVhdwu9Zpi/14MgyYz8GLscCGVR2rjbe4UIwZeFm9kje8+Dw6cYCChUn8Man81k8TFxBdWgUvaFngcXyv9NvaZcLo2Zj1wyZKM6XVJV4QJsPLqDE5twjDxPteBo+eeZ4+CBwSsLS5WDvrY515qHikeioz3SSSsOOCYoYDcroIow6xY4I51JvTX+jFj2rFuuzHpfUE0HQWEqoqDJ1JIU1nQcXdY7kQ1jJKT+i+fsKFfKU4TZaFezqBV1F+vHHz75uXARje3ECtl4TK7a0fNy6wZ55KTK0ySraVrIw1CMcV5RCTH0pKaKoWO5HZxKHGD1pOD8DS0xAilFLRZMJC1azIdbVBFUAF0EbIXEo80UNRkMgcNfEKojVOgGcgAB578vixp46fnn3u+OkzJ5+ZrCjtxw+kmElzOVDD74rpCKbw/wgdepU=')).decode()
assert hashlib.sha256(patch.encode()).hexdigest()=='c0a5a44ed3800368d5d8d99898a1d71d0d7e4116d4ae6f954d5d9bb93d2d06ac'

def make_checker(name,patched=False):
    module=types.ModuleType(name)
    sys.modules[name]=module
    ns=module.__dict__
    ns.update({k:globals()[k] for k in ['re','pd','np','torch','labse','util','nli_tok','nli_model','device','CONTRA']})
    for src in [frozen['model_functions'],*frozen['base_rules'],frozen['base_checker_functions'],frozen['merged_normalisation'],frozen['merged_checker']]:
        exec(src,ns)
    if patched: exec(patch,ns)
    assert (ns['SIM_T'],ns['NLI_T'])==(0.4,0.65)
    assert ns['RULE_SOURCE_HASHES']==frozen['source_hashes']
    return ns
baseline=make_checker('v7_frozen_comparison')
candidate=make_checker('v8_candidate_comparison',True)
started=datetime.now(timezone.utc).isoformat()
results=[]
Path('v8_500_partial.jsonl').write_text('')
print('Starting fresh V7 + V8 predictions: 500 notices on',device,flush=True)
for i,case in enumerate(inputs,1):
    item={'case_id':case['case_id']}
    for name,checker in [('frozen',baseline),('candidate',candidate)]:
        frame=checker['check_notice'](case['chinese'],case['english'])
        item[name]=json.loads(frame.to_json(orient='records',force_ascii=False))
    results.append(item)
    with Path('v8_500_partial.jsonl').open('a') as f:f.write(json.dumps(item,ensure_ascii=False)+'\n')
    if i%25==0:print(f'{i}/500 complete for BOTH versions',flush=True)
# Score only after every prediction has finished.
counts={name:Counter() for name in ['frozen','candidate']}
by_tier={name:{tier:Counter() for tier in ['routine','difficult','edge']} for name in counts}
for case,res in zip(references,results):
    assert case['case_id']==res['case_id']
    truth=case['expected_status']=='issue'
    for name in counts:
        flagged=any(r['why'] for r in res[name])
        outcome=('T' if truth==flagged else 'F')+('P' if flagged else 'N')
        counts[name][outcome]+=1
        by_tier[name][case['difficulty']][outcome]+=1
metadata=dict(started=started,completed=datetime.now(timezone.utc).isoformat(),device=str(device),
    dataset_sha256=hashlib.sha256(raw).hexdigest(),candidate_sha256=hashlib.sha256(patch.encode()).hexdigest(),
    baseline_source_sha256=hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),
    source_hashes=frozen['source_hashes'],thresholds=[0.4,0.65],
    python=platform.python_version(),packages={p:importlib.metadata.version(p) for p in ['torch','transformers','sentence-transformers']},
    model_ids=['sentence-transformers/LaBSE',NLI_NAME],shared_model_objects=True,
    evaluation='reused development benchmark; not held out for V8',counts=counts,by_difficulty=by_tier)
output=json.dumps(dict(metadata=metadata,results=results),ensure_ascii=False,indent=2)
Path('v8_500_results.json').write_text(output)
print('COMPARISON COMPLETE',flush=True)
for name in counts:
    print(name,dict(counts[name]),'accuracy',(counts[name]['TP']+counts[name]['TN'])/500)
    for tier,c in by_tier[name].items():print('  ',tier,dict(c),'accuracy',(c['TP']+c['TN'])/sum(c.values()))
print('Dataset SHA256:',metadata['dataset_sha256'])
print('Candidate SHA256:',metadata['candidate_sha256'])
# A compact, visible export of the complete raw outputs, not only aggregate scores.
export=base64.b64encode(zlib.compress(output.encode(),9)).decode()
display(HTML('<details><summary>Complete raw comparison export (compressed base64)</summary><textarea id="v8-result-export" rows="5" style="width:100%">'+export+'</textarea></details>'))
print('Raw results saved to v8_500_results.json')


In [ ]:
from IPython.display import HTML, display
summary_rows=[]
for tier in ['routine','difficult','edge']:
    a,b=by_tier['frozen'][tier],by_tier['candidate'][tier]
    n=sum(a.values()); positives=a['TP']+a['FN']
    summary_rows.append({'Difficulty':tier.title(),'Cases':n,
       'Accuracy: V7 → V8':f"{(a['TP']+a['TN'])/n:.1%} → {(b['TP']+b['TN'])/n:.1%}",
       'Errors caught: V7 → V8':f"{a['TP']}/{positives} → {b['TP']}/{positives}",
       'Missed: V7 → V8':f"{a['FN']} → {b['FN']}",
       'False alarms: V7 → V8':f"{a['FP']} → {b['FP']}"})
a,b=counts['frozen'],counts['candidate']
display(HTML('<h2>V8 versus frozen V7 — 500-case comparison</h2>'
    '<p><strong>Development results: these 500 cases informed V8. This is not a new held-out score.</strong></p>'
    +pd.DataFrame(summary_rows).to_html(index=False)
    +f"<p>Overall accuracy: <strong>{(a['TP']+a['TN'])/500:.1%} → {(b['TP']+b['TN'])/500:.1%}</strong>. "
    +f"Errors caught: {a['TP']}/150 → {b['TP']}/150. False alarms: {a['FP']}/350 → {b['FP']}/350.</p>"
    +'<p>Both versions ran fresh using the same model objects, original thresholds, and checksum-verified CSV. Original model weights were not fine-tuned.</p>'
    +'<p>To check another notice, call <code>candidate["check_notice"](chinese_text, english_text)</code>.</p>'))
